# Week 3, day 4 (morning) — Worksheet 09 SOLUTIONS: loading the fact table

Executed in the lab image. Every quoted number is what it actually printed.

Question 4 is the one to watch. The key lookup that resolves 2,382 enrollments to
their students fails for seven of them, and the `Unknown` member is the only thing
between that and a hole in the table.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 09 — Loading the fact table. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr, tx = load("enrollment"), load("transaction")
crs, prg, coh = load("course"), load("program"), load("cohort")
stu, cat, dtype = load("students"), load("category"), load("discount_type")


def surrogate(df, key_col, new_name, unknown=True):
    """Worksheet 08's dimension builder, reduced to just the key mapping."""
    out = (df[[key_col]].drop_duplicates().sort_values(key_col)
             .reset_index(drop=True))
    out[new_name] = range(1, len(out) + 1)
    out = out.rename(columns={key_col: "source_" + key_col})
    if unknown:
        out = pd.concat([pd.DataFrame([{"source_" + key_col: -1,
                                        new_name: -1}]), out],
                        ignore_index=True)
    return out


# The six dimension key maps, as worksheet 08 built them.
K_COURSE = surrogate(crs, "course_id", "course_id_sk")
K_PROGRAM = surrogate(prg, "program_id", "program_id_sk")
K_COHORT = surrogate(coh, "cohort_id", "cohort_id_sk")
K_STUDENT = surrogate(stu, "stu_id", "student_id_sk")
K_PROMO = surrogate(dtype, "discount_type_id", "promotion_id_sk")

# Worksheet 06's row filters R5 and R6 are NOT applied to the spine here --
# question 4 needs the orphans to still be present.
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
spine = enr[~enr.stu_id.isin(test_ids)].copy()

print("enrollment spine (TEST students removed):", len(spine))
print("key maps built:", ["course", "program", "cohort", "student", "promotion"])

PART A — resolving the keys

### Question 1

Start from the spine and resolve the two easy keys: `course_id` and `cohort_id`. Print the row count after each lookup and how many rows failed to resolve.
> **NOTE:** check the row count after every lookup. A dimension key that is not unique multiplies rows here, not later.

In [ ]:
fact = spine[["enrl_id", "enrl_date", "stu_id", "course_id", "cohort_id"]].copy()
print("spine:", len(fact))

for key_map, src, sk in [(K_COURSE, "course_id", "course_id_sk"),
                         (K_COHORT, "cohort_id", "cohort_id_sk")]:
    before = len(fact)
    fact = fact.merge(key_map, left_on=src, right_on="source_" + src,
                      how="left", validate="many_to_one")
    fact = fact.drop(columns=["source_" + src])
    print("  + %-12s %5d -> %5d rows, %d unresolved"
          % (src, before, len(fact), int(fact[sk].isna().sum())))

```
spine: 2382
  + course_id     2382 ->  2382 rows, 0 unresolved
  + cohort_id     2382 ->  2382 rows, 0 unresolved
```

Two lookups, nothing lost and nothing multiplied.

Every fact-table key lookup has exactly two ways to go wrong, and both are
checked on one line each:

**Rows multiply** if the dimension key is not unique. `validate="many_to_one"`
makes that impossible — worksheet 08 question 10 showed it catching a duplicated
dimension row before a single output row was produced. The row count printed
after each merge is the belt to that braces.

**Rows fail to resolve** if a fact references something not in the dimension.
`how="left"` keeps the row and leaves a null, so it is countable. An inner join
would have deleted it, and the row count would have quietly gone down — which
question 10 demonstrates.

Zero unresolved here because `course` and `cohort` are complete: every
`course_id` and `cohort_id` in `enrollment` exists in its source table. That is
worth confirming rather than assuming, and it is not true of every key — question
4 is the one that fails.

Note the spine is **2,382**, not 2,400: the 18 TEST-student enrollments are gone,
per worksheet 06's rule R5. The seven orphans are deliberately still here, because
question 4 needs them.

### Question 2

`program_id` is not on the enrollment at all — it has to come via `course`. Resolve it, and print how many enrollments got a program key.
> **NOTE:** worksheet 01 question 10 is why this needs two hops.

In [ ]:
fact = spine[["enrl_id", "course_id"]].copy()
fact = fact.merge(crs[["course_id", "program_id"]], on="course_id", how="left",
                  validate="many_to_one")
print("after course -> program_id:", len(fact), "rows")
print("null program_id:", int(fact.program_id.isna().sum()))

fact = fact.merge(K_PROGRAM, left_on="program_id",
                  right_on="source_program_id", how="left",
                  validate="many_to_one")
print()
print("after resolving the surrogate:", len(fact), "rows")
print("null program_id_sk:", int(fact.program_id_sk.isna().sum()))
print()
print(fact[["enrl_id", "course_id", "program_id", "program_id_sk"]]
      .head(3).to_string(index=False))

```
after course -> program_id: 2382 rows
null program_id: 0

after resolving the surrogate: 2382 rows
null program_id_sk: 0

 enrl_id  course_id  program_id  program_id_sk
  700001        203         103              3
  700002        202         102              2
  700003        222         106              6
```

Two hops for one key. The enrollment knows its course; the course knows its
program; only then can the surrogate be looked up.

This is worksheet 01 question 10's `KeyError` — *there is no `program_id` on an
enrollment* — resolved rather than raised. And it is a good illustration of what
the fact table is buying: three tables and two joins, done once at load time, so
that every future query filtering by program is a single join to `dim_program`.

Read the sample row: source course 203 belongs to source program 103, which is
surrogate key 3. Three different numbers for two different things, which is
exactly why the surrogate/source distinction from worksheet 08 question 9 has to
be maintained carefully.

`validate="many_to_one"` on both merges. The first asserts `course_id` is unique
in `course` — a property of the source, and the one that would silently inflate
the fact table if it broke. The second asserts the same of the key map.

**Zero nulls at both stages**, which means every course has a program and every
program is in the dimension. Worth printing even when it passes: it is the
before-and-after that makes a later regression visible.

### Question 3

`enrollment_date_id` is the `YYYYMMDD` integer from `dim_date`. Derive it from `enrl_date` and confirm every value falls inside the dimension's range.

In [ ]:
fact = spine[["enrl_id", "enrl_date"]].copy()
d = pd.to_datetime(fact.enrl_date)
fact["enrollment_date_id"] = d.dt.strftime("%Y%m%d").astype(int)

dim_dates = pd.date_range(pd.to_datetime(enr.enrl_date).min(),
                          pd.to_datetime(enr.enrl_date).max(), freq="D")
valid = set(dim_dates.strftime("%Y%m%d").astype(int))
print("dim_date rows:            ", len(valid))
print("distinct date keys in fact:", fact.enrollment_date_id.nunique())
print("keys not in dim_date:      ",
      int((~fact.enrollment_date_id.isin(valid)).sum()))
print()
print(fact.head(3).to_string(index=False))

```
dim_date rows:             675
distinct date keys in fact: 650
keys not in dim_date:       0

 enrl_id  enrl_date  enrollment_date_id
  700001 2024-09-09            20240909
  700002 2024-06-18            20240618
  700003 2025-04-07            20250407
```

The date key needs no join. `2024-09-09` becomes `20240909` by formatting, and
that integer *is* `dim_date.date_id`.

That is the payoff of worksheet 08's decision to use a meaningful date key rather
than a sequence. Every other dimension needs a lookup — a merge against a key map
that must exist first. The date key is computed from the fact row itself, which
means the fact load has no ordering dependency on `dim_date` at all.

**650 distinct keys against 675 dimension rows.** The 25 unused rows are the days
with no enrollments, and their presence is the point: a query joining
`fact_enrollment` to `dim_date` can find days with zero activity, because those
days exist in the dimension.

**`keys not in dim_date: 0`** is the check that matters. A fact row whose date
falls outside the dimension's range produces an orphan key — and this is the
easiest referential integrity failure to create, because it happens automatically
when data arrives past the date the dimension was generated to. It is why real
date dimensions are generated years ahead.

Note the assumption in `strftime("%Y%m%d")`: it requires `enrl_date` to have
parsed correctly. A date string that failed to parse becomes `NaT`, and `NaT`
formats to the string `"NaT"`, which `astype(int)` would reject — loudly, which is
the right outcome.

PART B — the lookup that fails

### Question 4

Resolve `student_id`. Print how many rows fail to match, then apply the `Unknown` member: unmatched rows get `student_id_sk = -1`. Print the count before and after.
> **NOTE:** worksheet 05 question 6 counted these. Confirm the number rather than trusting it.

In [ ]:
fact = spine[["enrl_id", "stu_id"]].copy()
fact = fact.merge(K_STUDENT, left_on="stu_id", right_on="source_stu_id",
                  how="left", validate="many_to_one")
unresolved = fact.student_id_sk.isna()
print("spine rows:            ", len(fact))
print("unresolved student_id: ", int(unresolved.sum()))
print()
print("the unmatched source ids:",
      sorted(int(v) for v in fact.loc[unresolved, "stu_id"].unique()))
print()
fact["student_id_sk"] = fact.student_id_sk.fillna(-1).astype(int)
print("after applying the Unknown member:")
print("  null student_id_sk:  ", int(fact.student_id_sk.isna().sum()))
print("  rows on student -1:  ", int((fact.student_id_sk == -1).sum()))
print("  rows lost:           ", len(spine) - len(fact))

```
spine rows:             2382
unresolved student_id:  7

the unmatched source ids: [5607, 5608, 5609, 5610, 5611, 5612, 5613]

after applying the Unknown member:
  null student_id_sk:   0
  rows on student -1:   7
  rows lost:            0
```

Seven enrollments reference students **5607 to 5613**, and none of those students
exist in `students`. Consecutive ids, which is itself a clue — this looks like a
block of records deleted from the source, or a range that was never extracted.

The `Unknown` member from worksheet 08 question 7 catches all seven. **Zero rows
lost**, and the anomaly is now a countable fact: `SELECT COUNT(*) FROM
fact_enrollment WHERE student_id = -1` returns 7, tonight and every night, and
worksheet 10 turns that into a monitored check.

Compare the three ways this could have gone:

| approach | rows kept | anomaly visible? |
|---|---|---|
| inner join | 2,375 | **no** — 7 rows silently gone |
| left join, null key | 2,382 | only if someone checks for nulls |
| left join, `Unknown` member | 2,382 | **yes** — 7 rows on student -1 |

The middle option is worse than it looks. A null foreign key means every query
joining to `dim_student` must remember `LEFT JOIN`, and the ones that use an
inner join lose the rows again — so the data is present in the fact table and
absent from every report built on it.

Note what the load does **not** do: guess. It does not invent a student record,
and it does not delete the enrollment. It records that seven enrollments happened
whose student is unknown, which is the truth.

The follow-up belongs to a human. Seven consecutive missing ids is a question for
whoever owns the source system, and it is answerable — the answer will be either
"we purged those students" (in which case the enrollments should probably have
gone too) or "the extract missed a range" (in which case the extract is broken).

### Question 5

Resolve `promotion_id`, which needs the enrollment-level promotion from `transaction`. Print how many enrollments get a real promotion and how many land on the `No Promotion` member.

In [ ]:
promo_per_enr = (tx.groupby("enrl_id").discount_type_id.max()
                   .rename("discount_type_id").reset_index())
fact = spine[["enrl_id"]].merge(promo_per_enr, on="enrl_id", how="left")
print("enrollments:                   ", len(fact))
print("with a promotion on their tx:  ", int(fact.discount_type_id.notna().sum()))
print("with none:                     ", int(fact.discount_type_id.isna().sum()))

fact["discount_type_id"] = fact.discount_type_id.fillna(-1)
fact = fact.merge(K_PROMO, left_on="discount_type_id",
                  right_on="source_discount_type_id", how="left",
                  validate="many_to_one")
print()
print("resolved promotion_id_sk:")
print(fact.promotion_id_sk.value_counts().sort_index()
        .rename("enrollments").to_string())

```
enrollments:                    2382
with a promotion on their tx:   1215
with none:                      1167

resolved promotion_id_sk:
promotion_id_sk
-1    1167
 1     180
 2     199
 3     234
 4     188
 5     219
 6     195
```

**1,167 of 2,382 enrollments — 49% — have no promotion**, and they land on
`promotion_id = -1`, the `No Promotion` member.

That is the single largest group in the distribution, which makes the design
decision concrete: without a member to point at, half the fact table would have a
null `promotion_id`, and half the fact table would vanish from any report that
joined `dim_promotion` with an ordinary inner join.

The five real promotions are evenly used — 180 to 234 enrollments each — which is
the sort of flat distribution that suggests generated data rather than a real
marketing programme, where one campaign usually dominates.

Two details in how the key was resolved.

**The promotion comes from `transaction`, not from `enrollment`.** It has to be
summarised to enrollment grain first — `groupby("enrl_id").max()` — because
`transaction` is at payment grain. Worksheet 07 question 2 confirmed
`discount_type_id` is constant within an enrollment, which is what makes `max()`
safe here. Had it varied, this key would not belong on this fact table at all.

**`fillna(-1)` before the merge, not after.** Setting the source key to -1 lets
the join itself resolve to the `Unknown` member, rather than joining, getting a
null, and patching it afterwards. Same result, one fewer step, and the intent is
visible in the code.

Note the distinction worksheet 08 question 7 drew: this member is labelled `No
Promotion`, not `Unknown`. These 1,167 enrollments genuinely had no promotion —
that is a business fact, not missing data. The seven from question 4 are the
other kind, and conflating them would hide which is which.

PART C — assembling the table

### Question 6

Build the measures at enrollment grain, deduplicating `transaction` first (worksheet 07 question 6), and attach them to the spine with a left join. Print the row count and the null count per measure.

In [ ]:
dupes = tx.drop(columns=["trans_id"]).duplicated()
clean = tx[~dupes]
print("transaction rows: %d, after dedupe: %d" % (len(tx), len(clean)))

per = clean.groupby("enrl_id").agg(
    tuition_amount=("full_price", "max"),
    amount_paid_to_date=("payment_amount", "sum"),
    discount_type_id=("discount_type_id", "max")).reset_index()
per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                on="discount_type_id", how="left")
per["discount_amount"] = per.discount_amount.fillna(0.0)

fact = spine[["enrl_id"]].merge(per.drop(columns=["discount_type_id"]),
                                on="enrl_id", how="left")
print()
print("fact rows:", len(fact), "(spine was %d)" % len(spine))
print()
for c in ["tuition_amount", "discount_amount", "amount_paid_to_date"]:
    print("  null %-22s %4d" % (c, int(fact[c].isna().sum())))

```
transaction rows: 4856, after dedupe: 4816

fact rows: 2382 (spine was 2382)

  null tuition_amount          156
  null discount_amount         156
  null amount_paid_to_date     156
```

The 40 duplicate rows from worksheet 07 question 6 are removed **before** the
aggregation, not after — which is the only order that works. Deduplicating a sum
afterwards is not possible; the information is gone.

**2,382 in, 2,382 out.** The left join attached measures without disturbing the
spine.

**156 rows of nulls**, and all three columns are null together — the enrollments
with no transaction at all. Worksheet 07 question 9 established what each of them
should default to, and they are not the same:

- `amount_paid_to_date` → **0**. True: no payments were made.
- `discount_amount` → **0**. True: no promotion was applied.
- `tuition_amount` → **not 0**. The course has a price; the warehouse does not
  know it. Defaulting to zero is what marked 156 non-payers as paid in full.

Question 7 applies the guard that stops that, and it is worth noting that the
guard is a patch. The correct fix is to source `tuition_amount` from the course
or cohort, where a price belongs, rather than from a payment row that may not
exist. **A measure that can only be learned from a transaction is a measure that
is missing for everyone who never transacted** — and those are usually the rows
you most want to see.

Printing the null count per measure before defaulting is the habit here. It is the
last moment at which the three columns are distinguishable; after `fillna` they
are all zeros and indistinguishable from real zeros.

### Question 7

Assemble the whole thing: all six keys, all five measures, the flag, in slide 38's column order. Print the shape, the columns, and three rows.
> **NOTE:** apply worksheet 07 question 9's guard so enrollments with no tuition are not marked paid in full.

In [ ]:
dupes = tx.drop(columns=["trans_id"]).duplicated()
per = (tx[~dupes].groupby("enrl_id")
       .agg(tuition_amount=("full_price", "max"),
            amount_paid_to_date=("payment_amount", "sum"),
            discount_type_id=("discount_type_id", "max")).reset_index())
per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                on="discount_type_id", how="left")

f = spine[["enrl_id", "enrl_date", "stu_id", "course_id", "cohort_id"]].copy()
f = f.merge(crs[["course_id", "program_id"]], on="course_id", how="left")
f = f.merge(per, on="enrl_id", how="left")
f["discount_type_id"] = f.discount_type_id.fillna(-1)

for km, left, drop in [(K_COURSE, "course_id", "source_course_id"),
                       (K_PROGRAM, "program_id", "source_program_id"),
                       (K_COHORT, "cohort_id", "source_cohort_id"),
                       (K_STUDENT, "stu_id", "source_stu_id"),
                       (K_PROMO, "discount_type_id", "source_discount_type_id")]:
    f = f.merge(km, left_on=left, right_on=drop, how="left").drop(columns=[drop])

f["enrollment_date_id"] = pd.to_datetime(f.enrl_date).dt.strftime("%Y%m%d").astype(int)
for c in ("tuition_amount", "discount_amount", "amount_paid_to_date"):
    f[c] = f[c].fillna(0.0)
f["net_tuition_amount"] = f.tuition_amount - f.discount_amount
f["enrollment_count"] = 1
f["is_paid_in_full"] = ((f.net_tuition_amount > 0)
                        & (f.amount_paid_to_date >= f.net_tuition_amount - 0.005)
                        ).astype(int)
for c in ["course_id_sk", "program_id_sk", "cohort_id_sk",
          "student_id_sk", "promotion_id_sk"]:
    f[c] = f[c].fillna(-1).astype(int)

# The natural keys have done their job; drop them before renaming the
# surrogates into their final names, or both end up called program_id.
f = f.drop(columns=["enrl_date", "stu_id", "course_id", "cohort_id",
                    "program_id", "discount_type_id"])
fact_enrollment = f.rename(columns={
    "enrl_id": "enrollment_id", "program_id_sk": "program_id",
    "course_id_sk": "course_id", "cohort_id_sk": "cohort_id",
    "student_id_sk": "student_id", "promotion_id_sk": "promotion_id",
})[["enrollment_id", "program_id", "course_id", "cohort_id", "student_id",
    "enrollment_date_id", "promotion_id", "enrollment_count",
    "tuition_amount", "discount_amount", "net_tuition_amount",
    "amount_paid_to_date", "is_paid_in_full"]]

print("fact_enrollment:", fact_enrollment.shape)
print("columns:", len(fact_enrollment.columns), "-- slide 38 lists 13")
print()
print(fact_enrollment.iloc[8:11].to_string(index=False))

```
fact_enrollment: (2382, 13)
columns: 13 -- slide 38 lists 13

 enrollment_id  program_id  course_id  cohort_id  student_id  enrollment_date_id  promotion_id  enrollment_count  tuition_amount  discount_amount  net_tuition_amount  amount_paid_to_date  is_paid_in_full
        700027           5         13          4         283            20240430             4                 1          5400.0           2000.0              3400.0              3400.00                1
        700028           8         24          6         361            20240803            -1                 1          5400.0              0.0              5400.0              5400.00                1
        700029           5         21          7         445            20240807            -1                 1          7200.0              0.0              7200.0              3839.73                0
```

**2,382 rows, 13 columns** — slide 38's specification, built.

Read the three rows. Enrollment 700027 used promotion 4 (`Scholarship`, 2,000 off
a 5,400 course) and paid the 3,400 net in full. 700028 paid list price in full.
700029 owes 7,200 and has paid 3,839.73, so the flag is 0.

Every one of those facts is now a single-table read. No joins, no rules, no
knowledge of which source column meant what. That is the whole return on the
previous eight worksheets.

Three things this final assembly gets right that are easy to get wrong.

**The natural keys are dropped.** `course_id` in this table is the surrogate 13,
not the source 203. Leaving both in — which the first version of this code did —
produces two columns called `course_id`, and a fact table where nobody can tell
which one a query should join on.

**The flag has a guard.** `(net_tuition_amount > 0) AND (paid >= net)`, from
worksheet 07 question 9. Without the first clause, the 156 enrollments with no
tuition would all read `is_paid_in_full = 1`.

**Every key is filled and cast to `int`.** `fillna(-1).astype(int)` — because a
float key is a silent bug waiting for a comparison, and a null key is a row that
disappears from an inner join.

The column order matches slide 38's listing: identifier, then the six dimension
keys, then the five measures, then the flag. Not required by anything, and worth
doing — a fact table read left to right should tell you *what happened*, then *how
much*.

### Question 8

Check the grain and the keys. Print whether `enrollment_id` is unique, the null count across the whole table, and how many rows sit on an `Unknown` member for each key.

In [ ]:
dupes = tx.drop(columns=["trans_id"]).duplicated()
per = (tx[~dupes].groupby("enrl_id")
       .agg(discount_type_id=("discount_type_id", "max")).reset_index())
f = spine[["enrl_id", "stu_id", "course_id", "cohort_id"]].copy()
f = f.merge(crs[["course_id", "program_id"]], on="course_id", how="left")
f = f.merge(per, on="enrl_id", how="left")
f["discount_type_id"] = f.discount_type_id.fillna(-1)
for km, left, drop in [(K_COURSE, "course_id", "source_course_id"),
                       (K_PROGRAM, "program_id", "source_program_id"),
                       (K_COHORT, "cohort_id", "source_cohort_id"),
                       (K_STUDENT, "stu_id", "source_stu_id"),
                       (K_PROMO, "discount_type_id", "source_discount_type_id")]:
    f = f.merge(km, left_on=left, right_on=drop, how="left").drop(columns=[drop])
for c in ["course_id_sk", "program_id_sk", "cohort_id_sk",
          "student_id_sk", "promotion_id_sk"]:
    f[c] = f[c].fillna(-1).astype(int)

print("rows:                  ", len(f))
print("enrollment_id unique:  ", f.enrl_id.is_unique)
print("total nulls:           ", int(f.isna().sum().sum()))
print()
print("rows sitting on an Unknown member:")
for c in ["course_id_sk", "program_id_sk", "cohort_id_sk",
          "student_id_sk", "promotion_id_sk"]:
    print("  %-18s %5d" % (c, int((f[c] == -1).sum())))

```
rows:                   2382
enrollment_id unique:   True
total nulls:            0

rows sitting on an Unknown member:
  course_id_sk           0
  program_id_sk          0
  cohort_id_sk           0
  student_id_sk          7
  promotion_id_sk     1167
```

Three assertions and a distribution, and the last two lines are the interesting
part.

**The grain holds.** `enrollment_id` unique confirms worksheet 02's corrected
grain — one row per enrollment event — survived nine worksheets of joins and
lookups. This is the check worksheet 02 question 10 argued for, and it is the one
that catches a fan-out from any of the five key lookups.

**Zero nulls.** Every key resolved, every measure defaulted. A null anywhere in a
fact table is a decision someone did not make.

**Then the Unknown counts, and they are two different stories.** Three keys have
zero — `course`, `program` and `cohort` are complete, and if that number ever
becomes non-zero, something upstream broke. Those three are the ones to alert on.

`student_id_sk` at **7** is a known, quantified anomaly with an open question
behind it. `promotion_id_sk` at **1,167** is expected business reality — half the
enrollments had no promotion.

Which is why this line belongs in the load's output every run rather than in a
one-off check. The number itself means nothing without a baseline; the *change*
means everything. Seven orphan students is a footnote. Seven hundred next Tuesday
is an incident, and the only way to notice is to have been printing seven.

**Record the expected value.** Worksheet 06's rule register did this for
standardization rules; the same idea applies to key resolution. `student_id = -1`
should be 7; anything else needs an explanation.

### Question 9

Reconcile against the source. Print the fact row count against the enrollment spine, and `SUM(amount_paid_to_date)` against the deduplicated `transaction` total for the same enrollments.
> **NOTE:** this is slide 39's row count check. It compares against something computed *outside* the fact table.

In [ ]:
dupes = tx.drop(columns=["trans_id"]).duplicated()
clean = tx[~dupes]
per = clean.groupby("enrl_id").payment_amount.sum().rename("amount_paid_to_date")
f = spine[["enrl_id"]].merge(per, on="enrl_id", how="left")
f["amount_paid_to_date"] = f.amount_paid_to_date.fillna(0.0)

print("enrollment spine rows:   ", len(spine))
print("fact_enrollment rows:    ", len(f))
print("match:                   ", len(spine) == len(f))
print()
in_scope = clean[clean.enrl_id.isin(set(spine.enrl_id))]
print("SUM(payment_amount) in source (in scope): %14.2f"
      % in_scope.payment_amount.sum())
print("SUM(amount_paid_to_date) in fact:         %14.2f"
      % f.amount_paid_to_date.sum())
print("difference:                               %14.2f"
      % (in_scope.payment_amount.sum() - f.amount_paid_to_date.sum()))

```
enrollment spine rows:    2382
fact_enrollment rows:     2382
match:                    True

SUM(payment_amount) in source (in scope):     8829541.87
SUM(amount_paid_to_date) in fact:             8829541.87
difference:                                         0.00
```

Two reconciliations, both exact, and both compare against something computed
**outside** the fact table.

That is what makes them worth running. Worksheet 07 question 10 built a fact table
that was internally perfect — unique grain, zero nulls — and missing 156 rows. No
check *within* the table could have found that. Only a comparison against the
spine it was supposed to be built from.

This is slide 39's first check: *"does `fact_enrollment` match the expected number
of valid enrollment records after filters are applied?"* Note the word
**expected**. 2,382 is not "a plausible number"; it is 2,400 minus the 18 TEST
enrollments, computable before the fact table exists.

**The money reconciliation is the stronger of the two**, because it is sensitive
to things the row count is not. A fan-out that duplicated rows would inflate the
sum. A wrong aggregation — `max` where `sum` was needed — would change it. A
filter applied in one place and not the other would show up. `difference: 0.00`
covers all of that in one line.

Two details that make it a real check rather than a ceremony.

**`in_scope` filters the source to the same enrollments.** Comparing the fact
total against the *whole* source would fail by design, since 18 TEST enrollments
were removed. A reconciliation has to compare like with like, and getting that
filter right is most of the work.

**It uses the deduplicated source.** Comparing against raw `transaction` would
show a difference of 62,324.35 — the duplicate rows from worksheet 07. Which
would be correct behaviour: the check *should* fail if the fact table silently
included them.

Run both, every load, and store the numbers. A reconciliation that has only ever
been run once tells you about one day.

### Question 10

Finally, load the fact table **without** the `Unknown` members: drop them from the key maps, resolve `student_id`, and assert no key is null. **This is supposed to fail.**

In [ ]:
no_unknown = K_STUDENT[K_STUDENT.student_id_sk != -1]
print("key map rows with Unknown:   ", len(K_STUDENT))
print("key map rows without Unknown:", len(no_unknown))
print()
f = spine[["enrl_id", "stu_id"]].merge(
    no_unknown, left_on="stu_id", right_on="source_stu_id", how="left")
print("fact rows:          ", len(f))
print("null student_id_sk: ", int(f.student_id_sk.isna().sum()))
print()
inner = spine[["enrl_id", "stu_id"]].merge(
    no_unknown, left_on="stu_id", right_on="source_stu_id", how="inner")
print("if the load used an INNER join instead: %d rows (%d lost, silently)"
      % (len(inner), len(spine) - len(inner)))
print()
assert f.student_id_sk.notna().all(), (
    "%d fact rows have an unresolved student_id and no Unknown member to "
    "point at" % int(f.student_id_sk.isna().sum()))

```
key map rows with Unknown:    607
key map rows without Unknown: 606

fact rows:           2382
null student_id_sk:  7

if the load used an INNER join instead: 2375 rows (7 lost, silently)

AssertionError: 7 fact rows have an unresolved student_id and no Unknown member to point at
```

One row removed from a dimension, and seven fact rows have nowhere to go.

The middle line is the one to sit with. **The left join did not fail.** It
produced 2,382 rows with seven null keys, no exception, no warning — a fact table
that will load into a warehouse, pass a row count check, and lose those seven rows
from every report that joins `dim_student` without remembering `LEFT`.

And the line below it is worse: **an inner join gives 2,375 rows and loses the
seven silently.** No nulls to find, because the rows are gone. The table is
smaller, internally consistent, and wrong — and the only way to notice is question
9's reconciliation against the spine.

Three defences, in the order they should be applied:

**The `Unknown` member**, which costs one row per dimension and makes this
impossible by construction.

**`how="left"` on every fact-to-dimension lookup**, so an unmatched row survives
as a null rather than being deleted.

**An assertion on the resulting key column**, because the first two are things
you did and this is a check that they worked.

**What this sheet established:**

| | |
|---|---|
| the load | 5 key lookups + 1 computed date key, every merge `many_to_one`, every merge `left` |
| `program_id` | two hops, because the source enrollment has no program at all |
| the date key | computed, not looked up — `20240909` **is** `dim_date.date_id` |
| unresolved students | **7** (ids 5607-5613), caught by the `Unknown` member, zero rows lost |
| no promotion | **1,167 of 2,382 — 49%** — on the `No Promotion` member |
| the finished table | **2,382 x 13**, grain unique, zero nulls |
| reconciliation | rows **2,382 = 2,382**, money **8,829,541.87 = 8,829,541.87**, difference **0.00** |
| without the Unknown member | 7 null keys, or 7 rows silently deleted by an inner join |

Worksheet 10 is Step 7: slide 39's five data-quality checks, run against this
table — and then the question the whole model exists to answer, which is whether
it can answer slide 25's six business questions.